# DevBench three-condition study planning

This notebook prepares configuration and an empty report. It does not generate data, download public datasets, fit models, or launch agent participants. The private repository must already be checked out in the Colab runtime; set `DEVBENCH_REPO` if its path differs. Keep the private plan and its blinding nonce out of public exports.


In [ ]:
# %% Environment configuration only
import os
from pathlib import Path
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[name] = "1"
repo = Path(os.environ.get("DEVBENCH_REPO", "/content/bio-agent-harness")).resolve()
if not (repo / "pyproject.toml").is_file():
    raise RuntimeError("Check out the repository first and set DEVBENCH_REPO to its absolute path. Do not paste login credentials into this notebook.")
print("Repository:", repo)


## Install the locked environment

This optional setup step installs packages only. It requires network access to the package registry; it does not acquire expression data. The separate project interpreter is used for planning so previously imported notebook libraries do not determine the project environment.


In [ ]:
# %% Package setup; no datasets or model services
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "sync", "--locked", "--extra", "reporting",
                "--directory", str(repo)], check=True)
project_python = repo / ".venv" / "bin" / "python"


In [ ]:
# %% Prepare only a proposed study and empty dashboard
from datetime import datetime, timezone
run_label = datetime.now(timezone.utc).strftime("study-plan-%Y%m%dT%H%M%S%fZ")
output_dir = repo / "results" / run_label
completed = subprocess.run([str(project_python), "-m", "devbench.study.cli", "plan",
    "--stage", "smoke", "--output", str(output_dir)], cwd=repo,
    check=True, text=True, capture_output=True)
import json
preparation = json.loads(completed.stdout)
assert preparation["data_executed"] is False
assert preparation["model_participants_launched"] == 0
print("Plan:", preparation["plan"])
print("Report:", preparation["report"])


In [ ]:
# %% Display the empty report, without scoring or executing data
from IPython.display import HTML, display
display(HTML(Path(preparation["report"]).read_text(encoding="utf-8")))


# Broad biology and public tracks

The following cells write configurations and empty reports only. The biological catalog has 97 cases; the public track contains two shared developmental panels and SEQC controls. Keep synthetic and public results separate. No run command or execution authorization is enabled.


In [ ]:
# %% Biological coverage plan only
biology_output = output_dir / "biology"
biology = subprocess.run([str(project_python), "-m", "devbench.study.cli", "biology-plan",
    "--coverage", "full", "--fresh-seeds", "--output", str(biology_output)],
    cwd=repo, check=True, text=True, capture_output=True)
biology_preparation = json.loads(biology.stdout)
assert biology_preparation["model_participants_launched"] == 0
assert biology_preparation["data_executed"] is False
display(HTML(Path(biology_preparation["report"]["paths"]["html"]).read_text()))


In [ ]:
# %% Public configuration and separate report only
public_output = output_dir / "public"
public = subprocess.run([str(project_python), "-m", "devbench.study.cli", "public-plan",
    "--output", str(public_output)], cwd=repo, check=True, text=True, capture_output=True)
public_preparation = json.loads(public.stdout)
assert public_preparation["model_participants_launched"] == 0
assert public_preparation["data_executed"] is False
display(HTML(Path(public_preparation["report"]).read_text()))


## Before a future benchmark

Select the exact model and assign independent generation seeds in a frozen execution plan. Review the prepared dataset provenance and the permission-boundary preflight. Dataset preparation and live runs have separate explicit authorization flags; none is enabled in this notebook. Codex login, supported operating-system isolation, and a working local tool endpoint must be verified in the eventual runtime. Colab availability alone does not establish those conditions.

The report currently shows **not run**. It is not evidence of biological accuracy, harness improvement, clinical validation, or regulatory compliance. See `docs/three-condition-study.md` and `reports/benchmark-redesign/research-review.md`.
